# Examples: API usage and RBF solution

This notebook contains two examples:
1. How to call the `/evaluate-with-visualization` endpoint from Python.
2. A complete solution for Problem C (RBF features) with gradient descent and animation.

In [ ]:
# Example: calling the API /evaluate-with-visualization endpoint
import requests
import numpy as np
import matplotlib.pyplot as plt

URL = "http://localhost:8000/evaluate-with-visualization"

quadratic_params = [0.0, 0.0, 0.0, -2.0, 2.0, 0.0]
payload = {"problem_id": "quadratic_binary", "x": quadratic_params}
r = requests.post(URL, json=payload)
r.raise_for_status()
data = r.json()
print('Loss:', data.get('y'))
print('Accuracy:', data.get('accuracy'))
coeffs = data.get('coefficients')
points = data.get('points')
if points and coeffs:
    points = np.asarray(points)
    xs = np.linspace(points[:,0].min()-0.5, points[:,0].max()+0.5, 200)
    ys = np.linspace(points[:,1].min()-0.5, points[:,1].max()+0.5, 200)
    XX, YY = np.meshgrid(xs, ys)
    grid = np.column_stack([XX.ravel(), YY.ravel()])
    b, w1, w2, w11, w22, w12 = coeffs
    Z = b + w1*grid[:,0] + w2*grid[:,1] + w11*grid[:,0]**2 + w22*grid[:,1]**2 + w12*grid[:,0]*grid[:,1]
    Z = Z.reshape(XX.shape)
    plt.figure(figsize=(6,5))
    plt.scatter(points[:,0], points[:,1], c=data.get('points') and np.asarray(points[:,0]) * 0, s=20)
    plt.contour(XX, YY, Z, levels=[0], colors=['k'])
    plt.title('Decision boundary from API coefficients')
    plt.show()


In [ ]:
# SOLUTION: Problem C — RBF features (9 centers -> 10 parameters incl bias)
import numpy as np
from app.datasets import get_dataset
from IPython.display import clear_output, display
import matplotlib.pyplot as plt
import time

X, y = get_dataset('quadratic_binary')

def rbf_features(X, centers, gamma=5.0):
    # squared euclidean distances -> (n, m)
    X = np.asarray(X)
    C = np.asarray(centers)
    d2 = np.sum((X[:, None, :] - C[None, :, :]) ** 2, axis=2)
    return np.exp(-gamma * d2)

def design_phi_rbf(X, centers, gamma=5.0):
    F = rbf_features(X, centers, gamma=gamma)
    ones = np.ones((F.shape[0],1))
    return np.hstack([ones, F])

def predict_proba_rbf(params, X, centers, gamma=5.0):
    p = np.asarray(params)
    Phi = design_phi_rbf(X, centers, gamma=gamma)
    logits = Phi @ p
    return 1.0 / (1.0 + np.exp(-logits))

def loss_rbf(params, centers, gamma=5.0, alpha=0.0):
    probs = predict_proba_rbf(params, X, centers, gamma=gamma)
    eps = 1e-12
    probs = np.clip(probs, eps, 1-eps)
    l = -np.mean(y * np.log(probs) + (1-y) * np.log(1-probs))
    if alpha and alpha>0:
        l += 0.5 * alpha * np.sum(params[1:]**2)
    return float(l)

def grad_rbf(params, centers, gamma=5.0, alpha=0.0):
    p = np.asarray(params)
    Phi = design_phi_rbf(X, centers, gamma=gamma)
    logits = Phi @ p
    probs = 1.0 / (1.0 + np.exp(-logits))
    errs = probs - y
    g = (Phi.T @ errs) / len(y)
    if alpha and alpha>0:
        g = g + np.concatenate([[0.0], p[1:]*alpha])
    return g

# choose centers on a 3x3 grid covering the data range
xmin, xmax = X[:,0].min()-0.2, X[:,0].max()+0.2
ymin, ymax = X[:,1].min()-0.2, X[:,1].max()+0.2
xs = np.linspace(xmin, xmax, 3)
ys = np.linspace(ymin, ymax, 3)
centers = np.array([[xx,yy] for xx in xs for yy in ys])
m = centers.shape[0]
print('Using', m, 'RBF centers -> params:', m+1)

# initialize params small
rng = np.random.default_rng(0)
params = rng.normal(scale=0.1, size=(m+1,))

history = [params.copy()]
lr = 0.8
for t in range(120):
    g = grad_rbf(params, centers, gamma=8.0, alpha=1e-3)
    params = params - lr * g
    history.append(params.copy())

# animation similar to previous helper: left=feature, right=loss slice over first weight
def plot_pair(p, centers, idx=(1,2)):
    fig, (axl, axr) = plt.subplots(1,2, figsize=(10,5))
    axl.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', s=20, edgecolor='k')
    xs = np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200)
    ys = np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 200)
    XX, YY = np.meshgrid(xs, ys)
    grid = np.column_stack([XX.ravel(), YY.ravel()])
    probs = predict_proba_rbf(p, grid, centers, gamma=8.0)
    ZZ = probs.reshape(XX.shape)
    axl.contour(XX, YY, ZZ, levels=[0.5], colors=['k'])
    axl.set_title('Feature space (RBF)')
    # loss landscape slice
    i,j = idx
    center_i, center_j = p[i], p[j]
    gv = np.linspace(center_i-2.0, center_i+2.0, 80)
    Gx, Gy = np.meshgrid(gv, gv)
    L = np.empty_like(Gx)
    for a in range(Gx.shape[0]):
        for b in range(Gx.shape[1]):
            q = p.copy()
            q[i] = Gx[a,b]
            q[j] = Gy[a,b]
            L[a,b] = loss_rbf(q, centers, gamma=8.0, alpha=1e-3)
    im = axr.imshow(L, extent=(gv.min(), gv.max(), gv.min(), gv.max()), origin='lower', cmap='viridis')
    axr.scatter([p[i]],[p[j]], color='red')
    fig.colorbar(im, ax=axr)
    plt.tight_layout()
    return fig

for p in history[::3]:
    fig = plot_pair(p, centers, idx=(1,2))
    clear_output(wait=True)
    display(fig)
    plt.close(fig)
    time.sleep(0.05)
